# Highbay Schema Prediction - MLP with EmbeddingBag

Objective: Train a lightweight Multi-Layer Perceptron (MLP) using PyTorch's `nn.EmbeddingBag` to classify schema fields and their associated sigils from natural language prose prompts. This model operates as a multi-label classifier to determine which schema attributes should be retrieved or added.

In [ ]:
# 1. Setup & Imports
import os
import json
import re
from collections import Counter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np

In [ ]:
# 2. Locate Dataset
# Check local paths first, fall back to Google Drive paths if run in Google Colab
LOCAL_PATH = "inputs/synthetic_typed_markdown_v5.jsonl"
DRIVE_PATH = "/content/drive/MyDrive/HighbayGeniusTraining/datasets/processed/synthetic_typed_markdown_v5.jsonl"

if os.path.exists(LOCAL_PATH):
    DATASET_PATH = LOCAL_PATH
elif os.path.exists(DRIVE_PATH):
    DATASET_PATH = DRIVE_PATH
else:
    # Attempting to mount Google Drive in Colab
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        DATASET_PATH = DRIVE_PATH
    except ImportError:
        DATASET_PATH = LOCAL_PATH
        print(f"Warning: Using fallback local path: {DATASET_PATH}")

print(f"Using dataset path: {DATASET_PATH}")

In [ ]:
# 3. Parse Target IR to Extract Schema Fields and ASCII Sigils
def map_sigil_to_ascii(sigil):
    if not sigil:
        return "[abc]"  # Default fallback
    sigil = sigil.strip()
    if sigil in ["⭘", "[⭘]"]:
        return "[o]"
    elif sigil in ["⚯", "[⚯]"]:
        return "[-]"
    elif "abc" in sigil:
        return "[abc]"
    elif "#" in sigil:
        return "[#]"
    elif "$" in sigil:
        return "[$]"
    return f"[{sigil}]"

def extract_schema_fields(ir_text):
    # Match standard bindings like: {{ [sigil] path }} or {{ sigil path }}
    # Also match repeats: source={{ [#] path }}
    pattern = r"\{\{\s*(?:\[([^\]]+)\]|([^\s\}]+))?\s*([a-zA-Z0-9_\(\)\.]+)\s*\}\}"
    matches = re.findall(pattern, ir_text)
    
    fields = set()
    for m in matches:
        sigil_opt1, sigil_opt2, path = m
        sigil = sigil_opt1 if sigil_opt1 else sigil_opt2
        ascii_sigil = map_sigil_to_ascii(sigil)
        fields.add(f"{ascii_sigil}{path}")
        
    # Match trigger calls as methods/effects if they specify attributes
    trigger_pattern = r"trigger\(([a-zA-Z0-9_\(\)\.]+)\)"
    triggers = re.findall(trigger_pattern, ir_text)
    for t in triggers:
        fields.add(f"[action]{t}")
        
    return list(fields)

# Load dataset
dataset = []
all_labels = set()

with open(DATASET_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        item = json.loads(line)
        prompt = item["prompt"]
        ir = item["target_ir"]
        fields = extract_schema_fields(ir)
        dataset.append({"prompt": prompt, "fields": fields})
        all_labels.update(fields)

label_list = sorted(list(all_labels))
label_to_idx = {label: i for i, label in enumerate(label_list)}

print(f"Loaded {len(dataset)} examples.")
print(f"Extracted {len(label_list)} unique schema fields:")
for label in label_list[:15]:
    print(f"  {label}")
if len(label_list) > 15:
    print("  ...")

In [ ]:
# 4. Tokenization & Vocab Building
def clean_and_tokenize(text):
    text = text.lower()
    # Replace punctuation with spaces
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return text.split()

# Build vocabulary
word_counts = Counter()
for item in dataset:
    tokens = clean_and_tokenize(item["prompt"])
    word_counts.update(tokens)

# Filter infrequent words and assign indices
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, count in word_counts.items():
    if count >= 1:  # Keep all words given small dataset size
        vocab[word] = len(vocab)

print(f"Vocabulary size: {len(vocab)} words")

In [ ]:
# 5. PyTorch Dataset and Collate FN for EmbeddingBag
class SchemaDataset(Dataset):
    def __init__(self, data, vocab, label_to_idx):
        self.data = data
        self.vocab = vocab
        self.label_to_idx = label_to_idx
        
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = clean_and_tokenize(item["prompt"])
        token_ids = [self.vocab.get(t, 1) for t in tokens]  # Fallback to <UNK> if word not in vocab
        
        # Multilabel classification vector
        target = np.zeros(len(self.label_to_idx), dtype=np.float32)
        for f in item["fields"]:
            if f in self.label_to_idx:
                target[self.label_to_idx[f]] = 1.0
                
        return torch.tensor(token_ids, dtype=torch.long), torch.tensor(target, dtype=torch.float32)

def collate_bag(batch):
    label_list, text_list, offsets = [], [], [0]
    for (_text, _label) in batch:
        label_list.append(_label)
        text_list.append(_text)
        offsets.append(offsets[-1] + len(_text))
        
    label_list = torch.stack(label_list)
    offsets = torch.tensor(offsets[:-1], dtype=torch.long)
    text_list = torch.cat(text_list)
    return text_list, offsets, label_list

In [ ]:
# 6. Define the EmbeddingBag MLP Model
class SchemaMLP(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super(SchemaMLP, self).__init__()
        # nn.EmbeddingBag computes the mean or sum of bag elements directly
        self.embedding = nn.EmbeddingBag(vocab_size, embed_dim, mode='mean')
        self.fc1 = nn.Linear(embed_dim, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)
        self.fc2 = nn.Linear(64, num_classes)
        
    def forward(self, text, offsets):
        embedded = self.embedding(text, offsets)
        x = self.relu(self.fc1(embedded))
        x = self.dropout(x)
        return self.fc2(x)

In [ ]:
# 7. Train the Model
EMBED_DIM = 64
BATCH_SIZE = 8
EPOCHS = 100
LEARNING_RATE = 0.01

train_dataset = SchemaDataset(dataset, vocab, label_to_idx)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_bag)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SchemaMLP(len(vocab), EMBED_DIM, len(label_list)).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(f"Training EmbeddingBag MLP on {device}...")
model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for text, offsets, targets in train_loader:
        text, offsets, targets = text.to(device), offsets.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(text, offsets)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * targets.size(0)
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1:02d}/{EPOCHS} - Loss: {total_loss / len(dataset):.4f}")

print("Training complete.")

In [ ]:
# 8. Inference / Prediction Seam
def predict_schema_fields(prompt, threshold=0.3):
    model.eval()
    tokens = clean_and_tokenize(prompt)
    token_ids = [vocab.get(t, 1) for t in tokens]
    
    text_tensor = torch.tensor(token_ids, dtype=torch.long).unsqueeze(0).to(device)
    offsets_tensor = torch.tensor([0], dtype=torch.long).to(device)
    
    with torch.no_grad():
        logits = model(text_tensor, offsets_tensor)
        probs = torch.sigmoid(logits).squeeze(0).cpu().numpy()
        
    predicted_fields = []
    for idx, prob in enumerate(probs):
        if prob >= threshold:
            predicted_fields.append((label_list[idx], float(prob)))
            
    # Sort by probability descending
    predicted_fields.sort(key=lambda x: x[1], reverse=True)
    return predicted_fields

# Test predictions
test_prompts = [
    "Create a checkout review step that displays user().email and cart().total.",
    "Design a simple app settings screen with a toggle for dark mode and push notifications.",
    "Show a profile banner image and details about the user like follower count."
]

print("\n=== TEST PREDICTIONS ===")
for prompt in test_prompts:
    print(f"\nPrompt: {prompt}")
    predictions = predict_schema_fields(prompt, threshold=0.2)
    if not predictions:
        print("  No fields predicted above threshold.")
    for field, score in predictions:
        print(f"  {field:<30} (Confidence: {score*100:.1f}%)")